[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [13]:
seq_len = 4

In [11]:
i = torch.arange(seq_len).unsqueeze(1)
j = torch.arange(seq_len).unsqueeze(0)

In [12]:
i - j

tensor([[ 0, -1, -2, -3],
        [ 1,  0, -1, -2],
        [ 2,  1,  0, -1],
        [ 3,  2,  1,  0]])

In [14]:
i = torch.arange(seq_len).unsqueeze(0)
j = torch.arange(seq_len).unsqueeze(1)

In [15]:
i - j

tensor([[ 0,  1,  2,  3],
        [-1,  0,  1,  2],
        [-2, -1,  0,  1],
        [-3, -2, -1,  0]])

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    # Q: (batch_size, seq_len, d_k)
    # K: (batch_size, seq_len, d_k)
    # V: (batch_size, seq_len, d_v)
    _, seq_len, d_k = Q.size()
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (batch_size, seq_len, seq_len)
    i = torch.arange(seq_len).unsqueeze(0)  # (1, seq_len)
    j = torch.arange(seq_len).unsqueeze(1)  # (seq_len, 1)
    masks = (i - j).abs() > window_size
    scores = scores.masked_fill(masks, float('-inf'))
    attn_weights = torch.softmax(scores, dim=-1)
    output = attn_weights @ V
    return output

In [4]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [5]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.9ms)
  ✅ [2/5] window_size=0 — only sees itself (0.3ms)
  ✅ [3/5] Large window equals full attention (4.6ms)
  ✅ [4/5] Distant tokens don't affect output (2.4ms)
  ✅ [5/5] Gradient flow (27.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (36.5ms total)
  Progress saved. Run status() to see your dashboard.

